In [ ]:
import time
import gc
import torch
import torch
from pathlib import Path
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_DIR = Path("/kaggle/input/datasets/user_name/gigachat_model")

searching_features = False

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map='auto',
    dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
)

tokenizer.padding_side = "left" 
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


device = next(model.parameters()).device
print(f"Model on: {device}")

In [ ]:
def ask_model(question, system_prompt="Я буду задавать тебе вопросы а ты будешь мне на них отвечать"):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.6,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    full_response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return full_response.strip()

In [ ]:
def ask_model_batch(questions, system_prompt="Я буду задавать тебе вопросы а ты будешь мне на них отвечать"):
    batch_messages = [
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": q}
        ] for q in questions
    ]
    
    input_texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]
    
    inputs = tokenizer(input_texts, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.6,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    input_len = inputs['input_ids'].shape[1]
    responses = [tokenizer.decode(output[input_len:], skip_special_tokens=True).strip() for output in outputs]
    
    return responses

In [ ]:
DATASET = pd.read_csv('dataset_path', encoding='utf-16')
print(len(DATASET))
DATASET = DATASET.drop_duplicates(subset=['prompt'], keep='first')
print(len(DATASET))

In [ ]:

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

clear_gpu()

In [ ]:
BATCH_SIZE = 2

SAVE_INTERVAL = 50 
DATASET['model_answer'] = DATASET['model_answer'].astype(object)
total_rows = len(DATASET)
start_time = time.time()

print(f"Запуск обработки. Всего строк: {total_rows}, Батч: {BATCH_SIZE}")

for i in range(0, total_rows, BATCH_SIZE):
    indices = DATASET.index[i:i + BATCH_SIZE]
    batch_prompts = DATASET.loc[indices, 'prompt'].tolist()
    
    try:
        batch_answers = ask_model_batch(batch_prompts)
        DATASET.loc[indices, 'model_answer'] = batch_answers
        
        current_done = i + len(indices)
        percent = (current_done / total_rows) * 100
        elapsed = time.time() - start_time
        
        print(f"[{percent:.1f}%] Обработано {current_done}/{total_rows} строк...", end='\r')
        
        if (i // BATCH_SIZE) % SAVE_INTERVAL == 0:
            DATASET.to_csv('working_result.csv', index=False, encoding='utf-16')
            print(f"Сохранение! Прошло времени: {elapsed/60:.1f} мин. Текущий индекс: {i}")
            
        torch.cuda.empty_cache()
        gc.collect()
        
    except Exception as e:
        print(f"Ошибка на индексе {i}: {e}")
        DATASET.to_csv('error_backup.csv', index=False, encoding='utf-16')
        torch.cuda.empty_cache()
        gc.collect()
        continue

DATASET.to_csv('FINAL_ANSWERS.csv', index=False, encoding='utf-16')
print(f"Готово! Все {total_rows} строк обработаны.")

In [ ]:
DATASET.to_csv('ANSWER_AND_QUESTIONS.csv', index=False, encoding='utf-16')